# 12 — Duplicates, Replace & Regex

Data cleaning essentials: deduplication, value replacement, and regex-powered transformations.
Every pattern here appears in production pipelines and interviews.

In [ ]:
import pandas as pd
import numpy as np
import re

np.random.seed(42)

# Create a dataset with deliberate duplicates and messy values
base_employees = pd.DataFrame({
    'emp_id':     [1001, 1002, 1003, 1004, 1005, 1001, 1003, 1006],  # 1001, 1003 duplicated
    'name':       ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'alice', 'Carol', 'Frank'],
    'department': ['Engineering', 'SALES', 'HR', 'engineering', 'Marketing', 'Engineering', 'HR', 'Finance'],
    'salary':     [90000, 85000, 70000, 110000, 75000, 90000, 70000, 95000],
    'city':       ['Mumbai', 'Delhi', 'Bangalore', 'Pune', 'Chennai', 'Mumbai', 'Bangalore', 'Kolkata'],
    'phone':      ['9876543210', '  8765432109  ', '7654321098', '6543210987', '5432109876',
                   '9876543210', '7654321098', '4321098765']
})

print(base_employees)

## 1. Detecting Duplicates

In [ ]:
# duplicated() — boolean mask
# Default: keep='first' — marks all duplicates EXCEPT the first occurrence
dup_mask = base_employees.duplicated()
print("Fully duplicated rows:")
print(base_employees[dup_mask])

In [ ]:
# duplicated on specific subset — find emp_id duplicates
dup_by_id = base_employees.duplicated(subset=['emp_id'])
print("Duplicate emp_ids:")
print(base_employees[dup_by_id])

In [ ]:
# keep='last' — mark all except the last occurrence
dup_keep_last = base_employees.duplicated(subset=['emp_id'], keep='last')
print("Duplicates (keep last):")
print(base_employees[dup_keep_last])

# keep=False — mark ALL occurrences of duplicates
dup_all = base_employees.duplicated(subset=['emp_id'], keep=False)
print("\nAll duplicate occurrences:")
print(base_employees[dup_all])

## 2. drop_duplicates()

In [ ]:
# Drop fully duplicated rows
clean = base_employees.drop_duplicates()
print(f"Original: {len(base_employees)}  After drop_duplicates(): {len(clean)}")

In [ ]:
# Drop by subset — keep first occurrence of each emp_id
unique_by_id = base_employees.drop_duplicates(subset=['emp_id'], keep='first')
print(f"Unique by emp_id: {len(unique_by_id)}")
print(unique_by_id)

In [ ]:
# Real-world pattern: keep the row with the most recent update
# Sort by a timestamp first, then drop_duplicates(keep='last')
records = pd.DataFrame({
    'customer_id': [101, 102, 101, 103, 102],
    'status':      ['active', 'pending', 'churned', 'active', 'active'],
    'updated_at':  pd.to_datetime(['2023-01-01', '2023-01-01', '2023-03-15', '2023-02-01', '2023-04-01'])
})

latest_records = (
    records
    .sort_values('updated_at')
    .drop_duplicates(subset=['customer_id'], keep='last')
    .reset_index(drop=True)
)
print("Latest record per customer:")
print(latest_records)

## 3. replace() — Value Substitution

In [ ]:
df = base_employees.copy()

# Replace specific values
df['department'] = df['department'].replace({
    'SALES':       'Sales',
    'engineering': 'Engineering',
    'ENGINEERING': 'Engineering',
})
print(df['department'].value_counts())

In [ ]:
# Replace in entire DataFrame (multiple columns)
df2 = pd.DataFrame({
    'status': ['Yes', 'No', 'yes', 'N', 'Y', 'no'],
    'active': ['True', 'False', 'true', 'false', '1', '0']
})

# Normalize all variations to True/False
df2_clean = df2.replace({'Yes': True, 'yes': True, 'Y': True, '1': True,
                         'No': False,  'no': False,  'N': False, '0': False,
                         'True': True, 'true': True,
                         'False': False, 'false': False})
print(df2_clean)

In [ ]:
# Replace NaN with a value
s = pd.Series([1.0, np.nan, 3.0, np.nan, 5.0])
print(s.replace(np.nan, 0))

# Same as fillna but more general
# replace() can also use regex

In [ ]:
# replace() with regex=True
salary_str = pd.Series(['$90,000', '$85,000', '$110,000', '$75,000'])

salary_num = (
    salary_str
    .str.replace(r'[\$,]', '', regex=True)
    .astype(int)
)
print(salary_num)

## 4. String Cleaning — The str Accessor for Normalization

In [ ]:
df = base_employees.copy()

# Normalize department names: strip whitespace + title case
df['department'] = df['department'].str.strip().str.title()

# Normalize name: strip + title case
df['name'] = df['name'].str.strip().str.title()

# Clean phone: remove leading/trailing whitespace + non-digits
df['phone'] = df['phone'].str.strip().str.replace(r'\D', '', regex=True)

print(df)

## 5. Regex-Based Replacement and Extraction

In [ ]:
# Real-world: clean messy product IDs
products = pd.Series([
    'PROD-001-V2',
    'PROD 002 v3',
    'product-003',
    'P-004-V1',
    'prod_005'
])

# Extract just the numeric part
product_num = products.str.extract(r'(\d+)', expand=False).astype(int)
print("Product numbers:")
print(product_num)

# Normalize: lowercase, replace separators with hyphen
normalized = products.str.lower().str.replace(r'[\s_]+', '-', regex=True)
print("\nNormalized:")
print(normalized)

In [ ]:
# Extract multiple groups from structured strings
order_ids = pd.Series(['ORD-2023-1001-IN', 'ORD-2024-2042-US', 'ORD-2023-3056-IN', 'ORD-2024-4001-UK'])

parsed = order_ids.str.extract(
    r'ORD-(?P<year>\d{4})-(?P<order_num>\d+)-(?P<country>[A-Z]+)'
)
print(parsed)
print(parsed.dtypes)

In [ ]:
# str.findall() — extract all matches (returns list per row)
text_col = pd.Series([
    'Contact: 9876543210, 8765432109',
    'Phone: 7654321098',
    'No contact info'
])

phones_found = text_col.str.findall(r'\d{10}')
print(phones_found)

## 6. Deduplication — Advanced Patterns

In [ ]:
# Pattern: deduplicate after normalizing strings
# Two entries might be the same person with slightly different name capitalization

raw = pd.DataFrame({
    'name':  ['Alice Smith', 'alice smith', 'Bob Jones', 'BOB JONES', 'Carol White'],
    'email': ['alice@company.com', 'alice@company.com', 'bob@company.com',
              'bob@company.com', 'carol@company.com'],
    'salary': [90000, 90500, 85000, 84000, 70000]  # slight salary discrepancy
})

# Normalize and dedup — keep highest salary version
deduped = (
    raw
    .assign(email_lower=lambda df: df['email'].str.lower())
    .sort_values('salary', ascending=False)
    .drop_duplicates(subset=['email_lower'], keep='first')
    .drop(columns=['email_lower'])
    .reset_index(drop=True)
)
print(deduped)

In [ ]:
# Large dataset — count duplicates before removing
np.random.seed(42)
n = 10_000
trans = pd.DataFrame({
    'txn_id':   np.random.choice(range(8000), n),  # intentional duplicates
    'customer': np.random.randint(1, 1000, n),
    'amount':   np.round(np.random.uniform(10, 5000, n), 2)
})

n_dups = trans.duplicated(subset=['txn_id']).sum()
print(f"Total rows:            {len(trans):,}")
print(f"Duplicate txn_ids:     {n_dups:,}")
print(f"Unique txns:           {trans['txn_id'].nunique():,}")

clean_trans = trans.drop_duplicates(subset=['txn_id'], keep='first')
print(f"After dedup:           {len(clean_trans):,}")

## 7. replace() vs map() vs str.replace() — When to Use Which

In [ ]:
dept = pd.Series(['Engineering', 'Sales', 'HR', 'Marketing', 'Finance'])

# replace(): great for value substitution (works on any dtype)
print(dept.replace({'HR': 'Human Resources', 'Sales': 'Revenue'}))

# map(): element-wise transform, dict lookup (unmapped → NaN)
dept_code = {'Engineering': 'ENG', 'Sales': 'SAL', 'HR': 'HRM'}
print(dept.map(dept_code))  # Marketing, Finance → NaN

# str.replace(): regex/literal on string Series only
print(dept.str.replace('ing', 'ING', regex=False))

## 8. Real-World Data Cleaning Pipeline

In [ ]:
raw_crm = pd.DataFrame({
    'Customer_ID':  [101, 102, 103, 101, 104, 102],
    'Name':         ['  Alice Smith ', 'BOB jones', 'carol WHITE', 'Alice Smith', 'Dave Brown', 'Bob Jones'],
    'Email':        ['alice@co.com', 'BOB@CO.COM', 'carol@co.com', 'alice@co.com', 'dave@co.com', 'bob@co.com'],
    'Revenue':      ['$12,500', '$8,900', '$15,200', '$12,500', None, '$9,100'],
    'Segment':      ['Premium', 'standard', 'PREMIUM', 'Premium', 'Standard', 'Standard'],
})

cleaned_crm = (
    raw_crm
    # Normalize column names
    .rename(columns=lambda c: c.strip().lower().replace(' ', '_'))
    # Clean string columns
    .assign(
        name    = lambda df: df['name'].str.strip().str.title(),
        email   = lambda df: df['email'].str.lower().str.strip(),
        segment = lambda df: df['segment'].str.title(),
        revenue = lambda df: pd.to_numeric(
            df['revenue'].str.replace(r'[\$,]', '', regex=True),
            errors='coerce'
        )
    )
    # Remove duplicates — keep first occurrence
    .drop_duplicates(subset=['customer_id', 'email'], keep='first')
    .reset_index(drop=True)
)

print(cleaned_crm)
print(cleaned_crm.dtypes)